# Tutorial: Finding disease-associated cell neighborhoods with scEPS

[scEPS](https://github.com/Genentech/sceps) (single-cell Expression exPlainability
Statistics) combines GWAS summary statistics with a multi-donor single-cell atlas to answer
one question: **in which cell neighborhoods does variation in disease-gene expression
explain variation in the disease phenotype across donors?**

A *cell neighborhood* is a local region of the k-NN graph, anchored on a single cell. In
each neighborhood, scEPS decomposes the variance of the donor-level phenotype into the part
attributable to GWAS genes, the part attributable to a set of **mean-expression-matched
control genes**, and the part attributable to all remaining genes. This gives, per
neighborhood:

$$
d = \omega^2_{gwas} - \omega^2_{ctrl}
$$

where $\omega^2_{gwas}$ is the phenotypic variance explained *per GWAS gene* and
$\omega^2_{ctrl}$ the same quantity per control gene. The control set matters: highly
expressed genes simply have more expression variance to offer, so comparing GWAS genes
against expression-matched controls is what makes $d$ interpretable. A positive,
significant $d$ says the neighborhood's disease-gene expression tracks the phenotype more
closely than chance expression variation of comparable magnitude would.

Note what scEPS consumes: **gene-level GWAS statistics plus expression**. It never touches
individual genotypes, so the genotype side of a `DonorData` object (`dd.G`) is used here
only as a source of donor metadata.

The method is described in the [preprint](https://www.medrxiv.org/content/10.64898/2026.06.26.26356714v1);
results for the diseases in the paper can be browsed in the
[result explorer](https://scepsresultexplorer.streamlit.app/).

## The workflow

scEPS ships as four command-line tools, one per step, preceded by two preparation steps
you do yourself:

| Step | Tool | What it produces |
| --- | --- | --- |
| 0a | MAGMA | gene-level association statistics (`GENE`, `ZSTAT`) from GWAS summary statistics |
| 0b | scanpy | a QC'd, batch-harmonized `.h5ad` with a k-NN graph |
| 1 | `sceps` | $d$ and the $\omega^2$ components for every cell neighborhood |
| 2 | `sceps-cluster-neighborhood` | approximately independent neighborhood blocks, for the block bootstrap |
| 3 | `sceps-aggregate` | average $d$ per cell type, with block-bootstrap standard errors |
| 4 | `sceps-corr` | correlation between $d$ and each gene's expression across neighborhoods |

This tutorial runs all four on the OneK1K dataset. Steps 0a and 0b are
*explained* rather than executed — MAGMA needs an external binary and reference panels, and
the full QC recipe is dataset-specific — but everything needed to make the data
scEPS-ready is executed, since scEPS cannot run without a k-NN graph.

```{warning}
Nothing below is a biological finding. OneK1K is a healthy population cohort with no
disease phenotype, and the gene-level GWAS statistics used here are a **placeholder** (see
Step 0a).
The purpose is to show the mechanics, the flags that matter, and the shape of each output
file. On real data, follow the checklist in the [summary](#summary-and-what-changes-for-a-real-study).
```

The [scEPS wiki](https://github.com/Genentech/sceps/wiki) is the reference for every option;
this tutorial covers the ones you need to get a first analysis out.

## Setup

scEPS is on PyPI and installs its own command-line tools:

```shell
pip install sceps
```

The optional preprocessing helper (`misc/preprocess_scdata.py`, shown in Step 0b) also
needs `harmonypy`, available as an extra: `pip install "sceps[preprocess]"`. This tutorial
does not require it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import yaml

import cellink
from cellink.resources import get_onek1k

sc.settings.verbosity = 1

# ---------------------------------------------------------------------------
# Everything you might want to scale lives here.
# ---------------------------------------------------------------------------
SEED = 0

# Step 0b: how much of the atlas to keep. OneK1K is 1.25M cells from 981 donors; scEPS
# analyzes one neighborhood per cell and its regression has one row per donor *pair*,
# so a tutorial-sized subsample of both axes is essential.
N_DONORS = 100  # donors kept (scEPS needs >= 20)
N_TOP_CELLTYPES = 6  # keep the most abundant cell types, so Step 3 has enough per type
N_CELLS = 5_000  # cells kept, spread across all retained donors
N_HVG = 2_000  # highly variable genes, flagged for reference (the graph uses X_harmony)
N_NEIGHBORS = 15

# Step 1: the phenotype to decompose, and donor-level covariates to adjust for.
PHENO = "age"
COVAR = "sex,sceps.prop_cells_local"
# Neighborhoods to analyze, of the N_CELLS available. At ~2.5 s each (measured at this
# donor/gene scale), 300 is roughly 12 minutes on one core. A real run does all of them.
N_NEIGHBORHOODS = 300
N_PROBE = 20  # neighborhoods in the timing probe run

# Step 2: number of approximately independent blocks (default 50 assumes many more cells).
N_KMEANS_CLUSTERS = 20

# Step 4: a gene must be expressed in at least this many cells to be correlated.
MIN_NUM_NONZERO = 50

ADATA_PATH = Path("sceps_input.h5ad")
GENE_LIST_PATH = Path("sceps_magma_placeholder.txt")
ONEK1K_DIR = Path("onek1k_data")
OUT_DIR = Path("sceps_out")
OUT_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)

Confirm the scEPS command-line tools are on `PATH`:

In [ ]:
!sceps --help | head -12

## Step 0a: gene-level GWAS statistics (explained)

scEPS needs to know which genes are implicated by the GWAS, and how strongly. It takes this
as a whitespace-delimited text file with two columns it actually reads:

```text
GENE       ZSTAT
```

`GENE` must use the **same identifiers as the single-cell data** (gene symbols or Ensembl
IDs), and `ZSTAT` is the gene-level association statistic. Everything else in the file is
ignored, so MAGMA's output can be passed through as-is once the identifiers are fixed.

[MAGMA](https://cncr.nl/research/magma/) produces this in two calls — map SNPs to genes
within a window, then aggregate SNP-level p-values per gene:

```shell
# 1. Annotate: map SNPs to genes using a window (10kb up/downstream by default)
magma \
    --annotate window=10,10 \
    --snp-loc  reference_panel.bim \
    --gene-loc gene_annotation.txt \
    --out      annotation

# 2. Gene-level association statistics
magma \
    --bfile      reference_panel \
    --pval       gwas_sumstats.txt use=SNP,P ncol=N \
    --gene-annot annotation.genes.annot \
    --out        gene_stats
```

This needs three external inputs: GWAS summary statistics with SNP IDs and p-values, an LD
reference panel matching the GWAS population, and a gene annotation file. The panel and
annotation are downloadable from the MAGMA page.

```{note}
MAGMA labels genes by **Entrez ID** by default. You must map these to the identifiers your
`.h5ad` uses, or the gene sets will silently come out empty.
```

cellink wraps this pipeline, so you can go from a GWAS Catalog accession to a MAGMA gene
file without leaving Python. The cell below is the real path — it is left unexecuted here
because it downloads the MAGMA binary and a reference panel. See the
[scDRS/seismic tutorial](./scdrs_seismic.ipynb) and the
[MAGMA tutorial](./magma_analysis_tutorial.ipynb) for a worked version.

In [ ]:
# --- The real Step 0a (not executed in this tutorial) -----------------------
# from cellink.resources import get_gwas_catalog_study_summary_stats
# from cellink.tl.external import run_magma_pipeline
#
# gwas_df = get_gwas_catalog_study_summary_stats("GCST90018926", genome_build="GRCh37")
#
# magma_file = run_magma_pipeline(
#     gwas_df,
#     output_prefix="trait",
#     col_mapping={"chromosome": "CHR", "base_pair_location": "BP", "p_value": "P"},
#     n_samples=667_504,
#     magma_bin="./magma/magma",
#     genome_build="GRCh37",
#     gene_id_type="ensembl",     # match the identifiers in adata.var_names
#     config_file="../../configs/magma.yaml",
#     ld_source="reference_panel",
#     reference_panel="EUR",
# )
# # -> "trait.genes.out", with GENE and ZSTAT columns, ready for --gene-list

Instead of running MAGMA, this tutorial builds a **placeholder** gene-level file, using the
same construction as scEPS's own `sceps-generate-test-data`: `ZSTAT ~ N(0, 1)` for every
gene, with a randomly chosen subset drawn from `N(10, 1)` to stand in for the GWAS hits.
That is enough to exercise the real gene-selection code path.

We build it in Step 0b, once we know which genes survive QC. To use real statistics
instead, point `GENE_LIST_PATH` at your MAGMA output and skip that cell.

## Step 0b: prepare the single-cell data

scEPS reads one `.h5ad` file and expects the following from it:

| Requirement | Why | Here |
| --- | --- | --- |
| ≥ 20 donors with phenotype variation | the variance decomposition is *across donors*; fewer donors and there is nothing to decompose | 981 donors, subsampled to 100 |
| a donor ID column in `adata.obs` | groups cells by donor | `donor_id` |
| a donor-level phenotype | the quantity whose variance is decomposed | `age` (see below) |
| a k-NN graph in `adata.obsp["connectivities"]` | defines the neighborhoods | built below with `sc.pp.neighbors` |
| a batch-harmonized cell embedding in `adata.obsm` | the graph should be built on it, and Step 2 clusters neighborhoods in it | `X_harmony`, shipped with the dataset |
| a cell type column in `adata.obs` | Step 3 aggregates by it | `celltype` |

Gene and cell identifiers can be left to the index: if `--gene-id-col` / `--cell-id-col`
are not given, scEPS uses `adata.var.index` / `adata.obs.index` and refers to them
internally as `sceps.gene_index` / `sceps.cell_index`. We rely on that throughout, so the
four steps agree on identifiers without any extra bookkeeping.

### Fetching OneK1K

`cellink.resources.get_onek1k` fetches OneK1K and returns a `DonorData` with both
modalities. Building the genotype side means a `vcf2zarr` pass over a genome-wide VCF plus
PLINK for filtering, pruning and kinship — and scEPS never uses genotypes. So we pass
`only_download=True` and hand it a copy of its own config filtered to the expression file,
which keeps the download to one 4.4 GB `.h5ad` instead of also pulling the VCF. Reading
cellink's config rather than hard-coding the URL means this keeps working if the URL
changes.

In [ ]:
onek1k_config = Path(cellink.__file__).parent / "resources" / "config" / "onek1k.yaml"
config = yaml.safe_load(onek1k_config.read_text())
print("files get_onek1k would fetch:", [f["filename"] for f in config["remote_files"]])

# Keep only the expression file; scEPS needs expression and donor metadata, not genotypes.
config["remote_files"] = [f for f in config["remote_files"] if f["filename"].endswith(".h5ad")]
expression_only_config = Path("onek1k_expression_only.yaml")
expression_only_config.write_text(yaml.safe_dump(config))

# only_download=True skips the genotype preprocessing (and so needs no PLINK). It returns
# nothing -- the files land under <data_home>/onek1k/.
get_onek1k(config_path=str(expression_only_config), data_home=str(ONEK1K_DIR), only_download=True)

onek1k_h5ad = ONEK1K_DIR / "onek1k" / "onek1k_cellxgene.h5ad"
print(f"\n{onek1k_h5ad} ({onek1k_h5ad.stat().st_size / 1e9:.2f} GB)")

The full atlas is 1.25M cells, so we open it in backed mode and only pull the subsample we
need into memory.

In [ ]:
full = sc.read_h5ad(onek1k_h5ad, backed="r")
print(full)

In [ ]:
celltype_key = "predicted.celltype.l2"
obs = full.obs

print("donors:", obs["donor_id"].nunique())
print("batches (pool_number):", obs["pool_number"].nunique())
print("cell types:", obs[celltype_key].nunique())
print(
    "\nage: ",
    obs.groupby("donor_id", observed=True)["age"].first().describe()[["count", "mean", "std", "min", "max"]].to_dict(),
)
print("sex: ", obs["sex"].value_counts().to_dict())
print("disease:", obs["disease"].value_counts().to_dict())

### The full preprocessing recipe (explained)

scEPS assumes the data is already QC'd and batch-harmonized, with the k-NN graph built on
the harmonized embedding. The scEPS repository ships an example recipe in
`misc/preprocess_scdata.py`, reproduced below. It is intentionally *not* executed here: QC
thresholds are dataset-specific, and the harmony step needs `harmonypy`.

In [ ]:
# --- misc/preprocess_scdata.py, for reference (not executed) ---------------
"""
import numpy as np
import scanpy as sc
import scanpy.external as sce

adata = sc.read_h5ad("<raw counts>.h5ad")

# Drop batches with too few cells
batch_nm = "<batch column>"
counts = adata.obs[batch_nm].value_counts()
adata = adata[adata.obs[batch_nm].isin(counts[counts >= 3].index)]

# QC and log-normalization
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs.pct_counts_mt < 5, :]
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Batch-harmonized embedding and k-NN graph
sc.pp.highly_variable_genes(adata, n_top_genes=2000, inplace=True)
sc.pp.pca(adata, use_highly_variable=True)
sce.pp.harmony_integrate(adata, batch_nm, basis="X_pca", max_iter_harmony=50)
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)
adata.write_h5ad("harmonized.h5ad")

# Optionally also save a copy with batch regressed out of the expression itself,
# and a copy with no adata.X at all (Steps 2 and 3 only need the graph, so the
# small file keeps their memory use down).
sc.pp.regress_out(adata, keys=[batch_nm])
adata.X = adata.X.astype(np.float32)
adata.write_h5ad("harmonized.regressout.h5ad")

del adata.X
adata.write_h5ad("harmonized.noX.h5ad")
"""

```{tip}
That recipe writes three files on purpose. Step 1 needs the expression matrix, but Steps 2
and 3 only need the graph and `obs`, so pointing them at the `.noX` copy is the easiest way
to cut memory on a large atlas.
```

### Minimal preparation for this tutorial

OneK1K already comes QC'd, with a batch-harmonized embedding (`X_harmony`) and a UMAP, so
the heavy parts of that recipe are done. What is left is: subsample, normalize, build the
graph on the harmonized embedding, and attach the phenotype.

Two axes need subsampling. Cells, because scEPS analyzes one neighborhood per cell; and
**donors**, because the method-of-moments regression has one row per donor *pair* — 100
donors give 4,950 rows, but all 981 would give 481,690, which is far slower per
neighborhood without teaching anything extra.

We also keep only the most abundant cell types, so Step 3 has enough neighborhoods per type
to aggregate over.

In [ ]:
rng = np.random.default_rng(SEED)

donors = rng.choice(obs["donor_id"].cat.categories.to_numpy(), size=N_DONORS, replace=False)
top_celltypes = obs[celltype_key].value_counts().head(N_TOP_CELLTYPES)
print(top_celltypes)

eligible = np.where(obs["donor_id"].isin(donors) & obs[celltype_key].isin(top_celltypes.index))[0]
keep = np.sort(rng.choice(eligible, size=min(N_CELLS, eligible.size), replace=False))
print(f"\n{eligible.size:,} eligible cells -> keeping {keep.size:,}")

adata = full[keep].to_memory()

# A clean column name, so Step 3's output file is not called
# "step3.predicted.celltype.l2.sceps.omega.celltype.txt".
adata.obs["celltype"] = adata.obs[celltype_key].astype(str)

cells_per_donor = adata.obs["donor_id"].value_counts()
cells_per_donor = cells_per_donor[cells_per_donor > 0]
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes, {cells_per_donor.size} donors")
print(
    f"cells per donor: min {cells_per_donor.min()}, median {int(cells_per_donor.median())}, max {cells_per_donor.max()}"
)

`X` in this file is raw counts, so normalize and log-transform it. Then drop genes
expressed in almost no cells and build the k-NN graph **on `X_harmony`**, the embedding
the dataset was published with.

Note that we flag highly variable genes but do *not* subset to them. scEPS needs the full
gene set: its third variance component is "all remaining genes", and control genes are
sampled across the whole expression range.

In [ ]:
print("X before normalization: max", float(adata.X.max()), "-- raw counts" if float(adata.X.max()) > 50 else "")

if float(adata.X.max()) > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, inplace=True)

# The graph defines the neighborhoods, so it must be built on the batch-harmonized
# embedding -- otherwise neighborhoods track batch rather than biology.
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, use_rep="X_harmony")

print(f"\n{adata.n_obs:,} cells x {adata.n_vars:,} genes retained")
print("k-NN graph:", adata.obsp["connectivities"].shape)
print("embeddings available:", list(adata.obsm.keys()))

### The phenotype

scEPS decomposes the variance of a **donor-level** phenotype. OneK1K is a healthy
population cohort — every cell is labelled `disease == "normal"` — so there is no disease
phenotype to use. We use donor `age` instead: real donor-level variation, which keeps the
mechanics honest even though pairing it with placeholder GWAS statistics makes the result
meaningless. In a real analysis this column is your disease phenotype: a quantitative
trait, or a case/control indicator (scEPS dummy-codes a two-level phenotype automatically).

`age` and `sex` are already per-cell columns in this object, constant within donor, which
is exactly what scEPS expects. Covariates named in `--covar` are regressed out of the
phenotype up front, so they must be **donor-level**; the one exception is
`sceps.prop_cells_local`, which scEPS computes per neighborhood and which is worth
including so a neighborhood's cell abundance does not masquerade as a phenotype effect.

In [ ]:
adata.obs["age"] = adata.obs["age"].astype(float)
adata.obs["sex"] = adata.obs["sex"].astype(str)

# Donors with a missing phenotype would be dropped by scEPS anyway; do it explicitly.
adata = adata[adata.obs["age"].notna()].copy()

donor_pheno = adata.obs.groupby("donor_id", observed=True)["age"].first()
print(f"{donor_pheno.size} donors with a phenotype")
print(
    f"{PHENO}: mean {donor_pheno.mean():.1f}, sd {donor_pheno.std():.1f}, "
    f"range {donor_pheno.min():.0f}-{donor_pheno.max():.0f}, {donor_pheno.nunique()} distinct values"
)
print("sex:", adata.obs.groupby("donor_id", observed=True)["sex"].first().value_counts().to_dict())

### The placeholder gene-level statistics

As flagged in Step 0a: `ZSTAT ~ N(0, 1)` for every retained gene, with 1,000 genes drawn
from `N(10, 1)` standing in for the GWAS hits. With `--auto-gene-selection`, scEPS converts
`ZSTAT` to a one-sided p-value, applies Benjamini-Hochberg, and keeps genes at FDR < 0.05,
clamped to between `--min-num-gwas-genes` (500) and `--max-num-gwas-genes` (2,000) — so
roughly those 1,000 genes should come back.

In [ ]:
n_placeholder_hits = 1_000

genes = adata.var_names.to_numpy()
zstat = rng.standard_normal(genes.size)
hit_idx = rng.choice(genes.size, size=min(n_placeholder_hits, genes.size), replace=False)
zstat[hit_idx] = rng.normal(loc=10.0, size=hit_idx.size)

gene_list = pd.DataFrame({"GENE": genes, "ZSTAT": zstat})
gene_list.to_csv(GENE_LIST_PATH, sep="\t", index=False)

print(gene_list.sort_values("ZSTAT", ascending=False).head())
print(f"\n{GENE_LIST_PATH}: {gene_list.shape[0]:,} genes")

Finally, write the object scEPS will read. The command-line tools take a file path, not an in-memory object.

In [ ]:
adata.write_h5ad(ADATA_PATH)
print(adata)

## Step 1: scEPS statistics per cell neighborhood

This is the step that does the work. For each cell in turn, `sceps`:

1. grows a neighborhood around that cell by a random walk on the k-NN graph, expanding until
   the neighborhood holds enough cells (`--min-num-cell`) and enough donors
   (`--min-num-donor`, `--min-frac-donor`);
2. builds the donor × gene expression matrix for that neighborhood;
3. selects the GWAS genes, and samples control genes matched on mean expression;
4. decomposes the phenotypic variance into GWAS / control / remaining components by method
   of moments, and bootstraps standard errors.

The flags below are the recommended defaults from the wiki:

| Flag | Why |
| --- | --- |
| `--auto-gene-selection` | select GWAS genes by MAGMA FDR rather than a fixed top-N |
| `--scale-pheno` | standardize the phenotype globally before decomposing |
| `--scale-pheno-neighborhood` | standardize again within each neighborhood |
| `--covar` | regress out donor covariates; include `sceps.prop_cells_local` |

### Sizing the run

Cost scales with the number of neighborhoods (one per cell), the number of donors, and the
number of genes. As a reference point, on simulated data at this scale — 100 donors, 3,000
cells, 20,000 genes, ~1,000 GWAS genes — one neighborhood takes about **2.5 s** with the
default bootstrap standard errors, in ~1.6 GB of memory. That is ~2 hours for 3,000
neighborhoods on a single core, which is why real runs are parallelized.

Rather than trusting that number, start with a small probe and measure it on *your* data.
`--start-idx` / `--stop-idx` select a contiguous slice of neighborhoods.

```{note}
scEPS exits without recomputing if its output file already exists, so re-running this cell
is cheap but also does nothing. Pass `--force-rerun` to actually recompute.
```

In [ ]:
!sceps \
    --adata {ADATA_PATH} \
    --donor-id-col donor_id \
    --gene-list {GENE_LIST_PATH} \
    --auto-gene-selection \
    --pheno {PHENO} \
    --scale-pheno \
    --scale-pheno-neighborhood \
    --covar "{COVAR}" \
    --start-idx 0 --stop-idx {N_PROBE} \
    --out {OUT_DIR}/step1

The output file name records the slice: `step1.<start>-<stop>.sceps.omega.txt.gz`, alongside
a `.log` file recording every argument used. The `ELAPSED_TIME` column gives the
per-neighborhood cost, which is what to extrapolate from.

In [ ]:
probe = pd.read_table(OUT_DIR / f"step1.0-{N_PROBE}.sceps.omega.txt.gz")

per_nbhd = probe["ELAPSED_TIME"].mean()
print(f"{per_nbhd:.2f} s per neighborhood")
print(f"-> {N_NEIGHBORHOODS} neighborhoods ~ {per_nbhd * N_NEIGHBORHOODS / 60:.0f} min")
print(f"-> all {adata.n_obs:,} cells ~ {per_nbhd * adata.n_obs / 3600:.1f} h on one core")
print()
print(
    probe[
        ["CELL", "NEIGHBORHOOD_SIZE", "NUM_DONOR", "NUM_GENE_GWAS", "NUM_GENE_CONTROL", "NUM_GENE_REST", "ELAPSED_TIME"]
    ].head()
)

Note `NUM_GENE_GWAS` and `NUM_GENE_CONTROL`: the control set is matched in size to the GWAS
set, and both can vary slightly between neighborhoods because genes are filtered per
neighborhood on their across-donor expression variance (`--min-expr-var-thres`,
`--max-expr-var-thres`).

### The tutorial run

Now analyze the rest of the slice. Because the probe already covered neighborhoods 0-20, we
run 20 onwards and end up with two output shards — which is exactly how a real, parallelized
run looks. Step 3 combines shards by glob, so nothing extra is needed to stitch them back
together.

Expect this to take roughly `N_NEIGHBORHOODS` × the per-neighborhood time printed above
(~12 minutes with the tutorial defaults).

In [ ]:
!sceps \
    --adata {ADATA_PATH} \
    --donor-id-col donor_id \
    --gene-list {GENE_LIST_PATH} \
    --auto-gene-selection \
    --pheno {PHENO} \
    --scale-pheno \
    --scale-pheno-neighborhood \
    --covar "{COVAR}" \
    --start-idx {N_PROBE} --stop-idx {N_NEIGHBORHOODS} \
    --out {OUT_DIR}/step1

In [ ]:
print(sorted(p.name for p in OUT_DIR.glob("step1.*")))

The `.log` file is a plain record of the arguments the run used — worth keeping alongside results:

In [ ]:
log = pd.read_table(OUT_DIR / f"step1.0-{N_PROBE}.log")
log[
    log["ARGUMENT"].isin(
        [
            "adata",
            "gene_list",
            "pheno",
            "covar",
            "auto_gene_selection",
            "scale_pheno",
            "scale_pheno_neighborhood",
            "min_frac_donor",
            "num_var_comp",
            "num_bootstrap_regression",
            "seed",
        ]
    )
]

### Reading the output

One row per cell neighborhood, 37 columns. The ones to look at first:

| Column | Meaning |
| --- | --- |
| `CELL` | the cell the neighborhood is anchored on |
| `NEIGHBORHOOD_SIZE`, `NUM_DONOR` | how many cells / donors the neighborhood ended up with |
| `OMEGA_GWAS`, `OMEGA_CONTROL`, `OMEGA_REST` | variance explained per GWAS / control / remaining gene |
| `OMEGA_DIFF` | the $d$ statistic: `OMEGA_GWAS - OMEGA_CONTROL` |
| `SE_OMEGA_DIFF`, `Z_OMEGA_DIFF`, `P_Z_OMEGA_DIFF` | standard error and test of $d \ne 0$ |

The remaining columns are per-neighborhood diagnostics (mean and variance of expression in
each gene set, gene counts, step size, elapsed time) and the same SE/Z/p triplet for each
$\omega^2$ component. The
[wiki](https://github.com/Genentech/sceps/wiki/Step-1.-Estimate-scEPS-statistics-at-each-individual-cell-neighborhood)
documents all of them.

In [ ]:
step1 = pd.concat(
    [pd.read_table(p) for p in sorted(OUT_DIR.glob("step1.*.sceps.omega.txt.gz"))],
    ignore_index=True,
)
print(f"{step1.shape[0]} neighborhoods x {step1.shape[1]} columns")

# Neighborhoods that cannot reach --min-num-cell / --min-num-donor / --min-frac-donor are
# dropped, so fewer rows than requested is normal -- but a big shortfall means the QC
# thresholds are too strict for this graph.
print(f"{step1.shape[0]} of {N_NEIGHBORHOODS} attempted neighborhoods passed QC")
print(
    f"neighborhood size: median {step1['NEIGHBORHOOD_SIZE'].median():.0f} cells, "
    f"{step1['NUM_DONOR'].median():.0f} donors "
    f"(random-walk step size: median {step1['STEP_SIZE'].median():.0f})"
)

step1[
    [
        "CELL",
        "NUM_DONOR",
        "NEIGHBORHOOD_SIZE",
        "OMEGA_GWAS",
        "OMEGA_CONTROL",
        "OMEGA_DIFF",
        "SE_OMEGA_DIFF",
        "Z_OMEGA_DIFF",
        "P_Z_OMEGA_DIFF",
    ]
].head()

```{important}
Sign matters. scEPS counts a neighborhood as disease-associated only when $d > 0$ **and**
$\omega^2_{gwas} > 0$ — disease genes must explain *more* than matched controls, and explain
something in absolute terms. A large negative $d$ is not a hit.
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(step1["OMEGA_DIFF"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].axvline(0, color="k", lw=1, ls="--")
axes[0].set(xlabel="$d$  (OMEGA_DIFF)", ylabel="cell neighborhoods", title="Per-neighborhood $d$")

axes[1].scatter(step1["OMEGA_CONTROL"], step1["OMEGA_GWAS"], s=8, alpha=0.5, color="#4C72B0")
lims = [
    min(step1["OMEGA_CONTROL"].min(), step1["OMEGA_GWAS"].min()),
    max(step1["OMEGA_CONTROL"].max(), step1["OMEGA_GWAS"].max()),
]
axes[1].plot(lims, lims, color="k", lw=1, ls="--")
axes[1].set(xlabel=r"$\omega^2_{ctrl}$", ylabel=r"$\omega^2_{gwas}$", title="GWAS vs matched control genes")

plt.tight_layout()
plt.show()

Points above the diagonal are neighborhoods where disease genes out-explain their
expression-matched controls. With placeholder GWAS statistics on a healthy cohort, they
should scatter symmetrically around it — which is the expected null result here.

Mapping $d$ back onto the embedding is how you would actually look for structure. We use the
UMAP the dataset ships with, so this is the same view the OneK1K authors published:

In [ ]:
adata.obs["sceps_d"] = adata.obs.index.map(step1.set_index("CELL")["OMEGA_DIFF"])
adata.obs["sceps_z"] = adata.obs.index.map(step1.set_index("CELL")["Z_OMEGA_DIFF"])

sc.pl.umap(adata, color="celltype", frameon=False)

# Symmetric colour limits, so that the midpoint of the diverging map really is d = 0.
for col in ["sceps_d", "sceps_z"]:
    lim = float(np.nanmax(np.abs(adata.obs[col].to_numpy(dtype=float))))
    sc.pl.umap(adata, color=col, cmap="RdBu_r", vmin=-lim, vmax=lim, na_color="lightgrey", frameon=False)

Grey cells are the neighborhoods outside the analyzed slice.

### Scaling up

A real analysis runs every cell neighborhood, which means parallelizing. scEPS offers two
ways, and they produce interchangeable output shards:

```shell
# Option 1: let scEPS split the work evenly across N jobs (array job friendly)
sceps ... --total-num-job 100 --job-idx ${SLURM_ARRAY_TASK_ID} --out out/step1

# Option 2: pick the slice yourself
sceps ... --start-idx 0 --stop-idx 5000 --out out/step1
```

Other options worth knowing when the run is too slow or too large:

| Option | Effect |
| --- | --- |
| `--use-analytical-stderr` | analytical instead of bootstrap standard errors (2.5 s → 1.4 s per neighborhood in the reference run above; the disattenuation bootstrap still runs, so it is not a 1000× saving) |
| `--cell-type-col` + `--focal-cell-type` | analyze one cell type at a time, e.g. one job per type |
| `--sample-frac` | down-sample the neighborhoods analyzed |
| `--force-rerun` | by default scEPS exits if its output already exists; this overrides that |
| `--verbose` | per-neighborhood progress |

```{note}
If you split Step 1 by cell type, Step 2 must still be run **once across all cells**. The
blocks it defines are a property of the graph, not of any one analysis.
```

## Step 2: approximately independent neighborhood blocks

Neighborhoods overlap: adjacent cells share cells and donors, so their $d$ statistics are
correlated. Averaging them in Step 3 with a naive bootstrap would treat those correlated
estimates as independent and understate the standard error.

`sceps-cluster-neighborhood` fixes this by mini-batch k-means clustering neighborhoods in
the embedding space, giving blocks of neighborhoods with similar donor composition. Step 3
then resamples **blocks** rather than individual neighborhoods.

Only the graph and the embedding are used, so this can run on a stripped-down `.h5ad` with
no `adata.X`. Two flags matter: `--neighbors-use-rep` names the embedding — it should be the
same harmonized embedding the graph was built on, here `X_harmony` (the default is
`X_pca_harmony`, the name `misc/preprocess_scdata.py` produces) — and
`--num-kmeans-cluster` sets the number of blocks, whose default of 50 assumes far more
cells than this subsample has.

In [ ]:
!sceps-cluster-neighborhood \
    --adata {ADATA_PATH} \
    --donor-id-col donor_id \
    --neighbors-use-rep X_harmony \
    --num-kmeans-cluster {N_KMEANS_CLUSTERS} \
    --out {OUT_DIR}/step2

In [ ]:
blocks = pd.read_table(OUT_DIR / "step2.txt.gz")
print(blocks.head())
print(f"\n{blocks.shape[0]:,} neighborhoods -> {blocks['sceps.neighborhood_cluster'].nunique()} blocks")
print("block sizes:", blocks["sceps.neighborhood_cluster"].value_counts().describe()[["min", "50%", "max"]].to_dict())

The mapping is keyed on `sceps.cell_index` — the cell identifier scEPS derived from
`adata.obs.index`, which is why leaving `--cell-id-col` unset everywhere keeps the steps
consistent.

## Step 3: aggregate to cell types

`sceps-aggregate` averages $d$ across groups of neighborhoods and computes standard errors
by bootstrapping the blocks from Step 2. `--prefix` is a glob over the Step 1 shards, so a
sharded run is combined here. Run it once per grouping you care about:

In [ ]:
# Per cell type
!sceps-aggregate \
    --prefix "{OUT_DIR}/step1.*.sceps.omega.txt.gz" \
    --adata {ADATA_PATH} \
    --neighborhood-clusters {OUT_DIR}/step2.txt.gz \
    --cell-type-col celltype \
    --out {OUT_DIR}/step3

# Across all analyzed neighborhoods (no --cell-type-col)
!sceps-aggregate \
    --prefix "{OUT_DIR}/step1.*.sceps.omega.txt.gz" \
    --adata {ADATA_PATH} \
    --neighborhood-clusters {OUT_DIR}/step2.txt.gz \
    --out {OUT_DIR}/step3

Both runs also write `step3.sceps.omega.txt.gz`: the Step 1 shards concatenated, with
FDR-significance flags added across neighborhoods. Step 4 takes that file as its input.

In [ ]:
print(sorted(p.name for p in OUT_DIR.glob("step3.*")))

by_celltype = pd.read_table(OUT_DIR / "step3.celltype.sceps.omega.celltype.txt")
overall = pd.read_table(OUT_DIR / "step3.sceps.omega.celltype.txt")

cols = [
    "CELL_TYPE",
    "NUM_CELL",
    "MEAN_NUM_DONOR",
    "MEAN_OMEGA_DIFF",
    "SE_MEAN_OMEGA_DIFF",
    "Z_MEAN_OMEGA_DIFF",
    "P_Z_MEAN_OMEGA_DIFF",
]
by_celltype[cols].sort_values("Z_MEAN_OMEGA_DIFF", ascending=False)

In [ ]:
overall[cols]

The key columns mirror Step 1, one level up: `MEAN_OMEGA_DIFF` with `SE_`, `Z_` and `P_Z_`
prefixes. In addition, `NUM_SIGNIF_FDR{5,10,20}_OMEGA_DIFF` count how many *individual*
neighborhoods in the group were significant at each FDR threshold — useful for spotting a
cell type where a minority of neighborhoods carry the signal and the mean washes it out.

In [ ]:
signif_cols = [
    "CELL_TYPE",
    "NUM_CELL",
    "NUM_SIGNIF_FDR5_OMEGA_DIFF",
    "NUM_SIGNIF_FDR10_OMEGA_DIFF",
    "NUM_SIGNIF_FDR20_OMEGA_DIFF",
]
by_celltype[signif_cols].sort_values("NUM_SIGNIF_FDR20_OMEGA_DIFF", ascending=False)

A forest plot of the per-cell-type estimates is usually the headline figure:

In [ ]:
plot_df = by_celltype.sort_values("MEAN_OMEGA_DIFF")
y = np.arange(plot_df.shape[0])

fig, ax = plt.subplots(figsize=(7, 0.5 * plot_df.shape[0] + 1.5))
ax.errorbar(
    plot_df["MEAN_OMEGA_DIFF"],
    y,
    xerr=1.96 * plot_df["SE_MEAN_OMEGA_DIFF"],
    fmt="o",
    color="#4C72B0",
    ecolor="#4C72B0",
    capsize=3,
    lw=1,
)
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_yticks(y)
ax.set_yticklabels([f"{ct}  (n={n:,})" for ct, n in zip(plot_df["CELL_TYPE"], plot_df["NUM_CELL"], strict=True)])
ax.set(xlabel="mean $d$  (95% CI, block bootstrap)", title="Aggregated scEPS $d$ by cell type")
plt.tight_layout()
plt.show()

On real data, cell types whose interval sits clearly above zero are the disease-associated
populations. Here the intervals straddle (or sit below) zero, as they should.

Other options for this step:

| Option | Effect |
| --- | --- |
| `--cell-type-col a,b,c` | aggregate at several resolutions at once, one output file each |
| `--use-inverse-variance-weights` | weight neighborhoods by precision instead of averaging equally |
| `--use-neighborhoods` | aggregate over an explicit list of cell IDs |
| `--add-column NAME VALUE` | stamp a constant column onto the output, handy when merging many runs |
| `--num-bootstrap` | bootstrap replicates (1,000 by default) |

Omitting `--neighborhood-clusters` falls back to an ordinary bootstrap over individual
neighborhoods, which will generally give standard errors that are too small.

## Step 4 (optional): what distinguishes the associated neighborhoods?

`sceps-corr` correlates each gene's expression with the scEPS statistics across
neighborhoods (Spearman by default). It answers a follow-up question: having found where the
signal is, which genes mark those neighborhoods?

`--min-num-nonzero` drops genes expressed in too few cells; the default of 100 assumes far
more cells than this subsample has.

In [ ]:
# Across all analyzed neighborhoods
!sceps-corr \
    --adata {ADATA_PATH} \
    --sceps-result {OUT_DIR}/step3.sceps.omega.txt.gz \
    --min-num-nonzero {MIN_NUM_NONZERO} \
    --out {OUT_DIR}/step4

In [ ]:
step4 = pd.concat(
    [pd.read_table(p) for p in sorted(OUT_DIR.glob("step4.0-*.txt.gz"))],
    ignore_index=True,
).dropna(subset=["R_OMEGA_DIFF"])

print(f"{step4.shape[0]:,} genes correlated\n")
print("Most positively correlated with d:")
print(
    step4.nlargest(8, "R_OMEGA_DIFF")[["GENE", "R_OMEGA_GWAS", "R_OMEGA_CONTROL", "R_OMEGA_DIFF"]].to_string(
        index=False
    )
)
print("\nMost negatively correlated with d:")
print(
    step4.nsmallest(8, "R_OMEGA_DIFF")[["GENE", "R_OMEGA_GWAS", "R_OMEGA_CONTROL", "R_OMEGA_DIFF"]].to_string(
        index=False
    )
)

The same restricted to one cell type, via `--cell-type-col` / `--focal-cell-type`:

In [ ]:
focal = by_celltype.sort_values("Z_MEAN_OMEGA_DIFF", ascending=False)["CELL_TYPE"].iloc[0]
print("focal cell type:", focal)

!sceps-corr \
    --adata {ADATA_PATH} \
    --sceps-result {OUT_DIR}/step3.sceps.omega.txt.gz \
    --cell-type-col celltype \
    --focal-cell-type "{focal}" \
    --min-num-nonzero {MIN_NUM_NONZERO} \
    --out {OUT_DIR}/step4.focal

In [ ]:
step4_focal = pd.concat(
    [pd.read_table(p) for p in sorted(OUT_DIR.glob("step4.focal.*.txt.gz"))],
    ignore_index=True,
).dropna(subset=["R_OMEGA_DIFF"])

step4_focal.nlargest(8, "R_OMEGA_DIFF")[["GENE", "R_OMEGA_GWAS", "R_OMEGA_CONTROL", "R_OMEGA_DIFF"]]

```{warning}
By default `sceps-corr` reports correlations with **no** test statistics, deliberately: both
the scEPS statistics and gene expression are correlated across neighborhoods, which makes a
valid test hard to construct. Adding `--num-bootstrap 1000 --neighborhood-clusters
<step2 output>` will produce `SE_`, `Z_` and `P_` columns via block bootstrap, but treat
those p-values as descriptive rather than calibrated.
```

## Summary and what changes for a real study

The whole workflow, as four commands:

```shell
# Step 1 -- per-neighborhood statistics (parallelize over --job-idx)
sceps \
    --adata harmonized.h5ad \
    --donor-id-col donor_id \
    --gene-list trait.genes.out \
    --auto-gene-selection \
    --pheno disease_status \
    --scale-pheno --scale-pheno-neighborhood \
    --covar "sex,age,sceps.prop_cells_local" \
    --total-num-job 100 --job-idx ${SLURM_ARRAY_TASK_ID} \
    --out out/step1

# Step 2 -- approximately independent blocks (once, across ALL cells)
sceps-cluster-neighborhood \
    --adata harmonized.noX.h5ad \
    --donor-id-col donor_id \
    --neighbors-use-rep X_pca_harmony \
    --out out/step2

# Step 3 -- aggregate to cell types
sceps-aggregate \
    --prefix "out/step1.*.sceps.omega.txt.gz" \
    --adata harmonized.noX.h5ad \
    --neighborhood-clusters out/step2.txt.gz \
    --cell-type-col celltype \
    --out out/step3

# Step 4 -- correlate with gene expression (optional)
sceps-corr \
    --adata harmonized.h5ad \
    --sceps-result out/step3.sceps.omega.txt.gz \
    --out out/step4
```

The wiki suggests one directory per step, which keeps the globs in Steps 3 and 4 simple:

```text
DISEASE/
├── step_0a/out/     # MAGMA gene-level statistics
├── step_0b/out/     # harmonized .h5ad files
├── step1/out/       # one shard per parallel job
├── step2/out/
├── step3/out/
└── step4/out/
```

### Checklist for real data

- **Real gene-level statistics.** Replace the placeholder with MAGMA output, and check the
  gene identifiers match `adata.var_names` — a mismatch yields an empty gene set, not an error.
- **A real disease phenotype**, with variation across at least 20 donors. If it lives outside
  the `.h5ad`, pass it with `--pheno-file` (a table keyed by the same donor ID column) rather
  than merging it in by hand.
- **Harmonize before building the graph**, and point `--neighbors-use-rep` at the harmonized
  embedding.
- **Donor-level covariates** in `--covar`, and keep `sceps.prop_cells_local` in the list.
- **All cells**, via `--total-num-job` / `--job-idx` as an array job.
- **Bootstrap standard errors** (the default). Reach for `--use-analytical-stderr` only when
  compute is the binding constraint.
- **Interpret the sign**: a hit needs $d > 0$ *and* $\omega^2_{gwas} > 0$.

And the caveat once more: the numbers produced above are not findings. OneK1K is a healthy
cohort analyzed with `age` as a stand-in phenotype, the GWAS statistics were placeholders,
and only a subsample of donors, cells and neighborhoods was used. See the
[wiki](https://github.com/Genentech/sceps/wiki) for the full option
reference and the [preprint](https://www.medrxiv.org/content/10.64898/2026.06.26.26356714v1)
for the method.